In [ ]:
lazy import torch
lazy import numpy as np

In [ ]:
class biRNN(torch.nn.Module):
    def __init__(self, vocab_size: int, embedding_size:int, hidden_size: int, output_size: int, layers: int = 1, dropout: float = 0.2):
        super().__init__()

        self.embedding = torch.nn.Embedding(vocab_size, embedding_size)

        self.layers = layers
        self.hidden_size = hidden_size
        self.output_size = output_size

        self.Wxh_forward = torch.nn.ParameterList()
        self.Whh_forward = torch.nn.ParameterList()
        self.bh_forward = torch.nn.ParameterList()

        self.Wxh_backward = torch.nn.ParameterList()
        self.Whh_backward = torch.nn.ParameterList()
        self.bh_backward = torch.nn.ParameterList()
        
        self.layer_norms = torch.nn.ModuleList() 

        for i in range(layers):
            current_input_size = embedding_size if i == 0 else hidden_size * 2

            self.Wxh_forward.append(torch.nn.Parameter(torch.randn(current_input_size, hidden_size) * 0.01))
            self.Whh_forward.append(torch.nn.Parameter(torch.randn(hidden_size, hidden_size) * 0.01))
            self.bh_forward.append(torch.nn.Parameter(torch.zeros(hidden_size)))

            self.Wxh_backward.append(torch.nn.Parameter(torch.randn(current_input_size, hidden_size) * 0.01))
            self.Whh_backward.append(torch.nn.Parameter(torch.randn(hidden_size, hidden_size) * 0.01))
            self.bh_backward.append(torch.nn.Parameter(torch.zeros(hidden_size)))
            
            self.layer_norms.append(torch.nn.LayerNorm(hidden_size * 2))

        self.classifier = torch.nn.Sequential(
            torch.nn.Linear(hidden_size * 2, hidden_size),
            torch.nn.ReLU(),
            torch.nn.Dropout(dropout),
            torch.nn.Linear(hidden_size, output_size)
        )

    def forward(self, x):

        x = self.embedding(x)
        
        batch_size = x.size(0)
        seq_len = x.size(1)

        for layer in range(self.layers):
            h_forward = torch.zeros(batch_size, self.hidden_size, device=x.device)
            h_backward = torch.zeros(batch_size, self.hidden_size, device=x.device)

            forward_output = []
            backward_output = []

            for t in range(seq_len):
                xt = x[:, t, :]
                h_forward = torch.tanh(xt @ self.Wxh_forward[layer] + h_forward @ self.Whh_forward[layer] + self.bh_forward[layer])
                forward_output.append(h_forward)

            for t in reversed(range(seq_len)):
                xt = x[:, t, :]
                h_backward = torch.tanh(xt @ self.Wxh_backward[layer] + h_backward @ self.Whh_backward[layer] + self.bh_backward[layer])
                backward_output.append(h_backward)

            backward_output.reverse()

            f_out_tensor = torch.stack(forward_output, dim=1)
            b_out_tensor = torch.stack(backward_output, dim=1)

            x = torch.cat([f_out_tensor, b_out_tensor], dim=-1)
            
            x = self.layer_norms[layer](x)

        y = self.classifier(x)

        return y